# Milestone 3 — Design an Evaluation Strategy
# Scenario:
## Build a card-transaction fraud detector. Fraud accounts for 0.1% of transactions; chargeback labels arrive 30–60 days later, and fraud patterns change over time.

# Required analysis:

# 1 Identify which model comparison metrics to use and avoid, with justification.

    Fraud accounts for only 0.1% of transactions, meaning approximately 1 out of every 1,000 transactions is fraudulent. Therefore,     accuracy is misleading because a model that predicts every transaction as legitimate achieves 99.9% accuracy but detects no fraud.

# Metrics I would use:
    - Precision: Measures how many transactions flagged as fraud are actually fraud. High precision reduces unnecessary transaction 
    blocks and investigations.
    
    - Recall (fraud detection rate): Measures how much actual fraud the model catches. It is important because missing fraud costs an average of $200 per transaction.
    
    - PR-AUC (Precision–Recall AUC): Useful for comparing models on highly imbalanced data because it focuses on fraud detection and precision-recall trade-offs.
    
    - F1-score or F-beta score: F1 balances precision and recall. F2 gives more weight to recall when missing fraud is especially costly.

# Metrics I would avoid or not use alone:
    - Accuracy: Can look excellent even when the model misses every fraudulent transaction.
    
    - ROC-AUC alone: Can hide poor precision at the very low false-positive rates required for fraud detection.




# 2 Design the train/validation/test split.
I will split the data based on time:
- 70% Training: To train the model on older transactions.
- 15% Validation: To select the best model and decision threshold.
- 15% Testing: To check the final performance on newer transactions.

Imagine we are in October and building a fraud detection model using transactions from January to October.


| Dataset | Time period | Description |
| :--- | :--- | :--- |
| **Train** | Jan 1 – Jun 30 | |
| **Buffer** | Jul 1 – Aug 31 | Skip these transactions during training and tuning because their labels may still be pending. |
| **Validation**| Sep 1 – Sep 30 | Make predictions first, then reveal the labels to select the best model and threshold. |
| **Test** | Oct 1 – Oct 31 | Make predictions and evaluate the final model on newer transactions. Do not use this data to tune the model. |

# 3. Evaluation with delayed labels
    Since chargeback labels take 30–60 days to arrive, I cannot immediately know whether recent predictions are correct.

    I will evaluate older transactions with mature labels and monitor recent transactions while waiting for their labels.
    When new labels arrive, I will update the evaluation metrics. Transactions without labels will not automatically be considered normal.

# 4. Cost-Based Decision Threshold and Score Calibration
# A. Choose the decision threshold

        False Negative (FN): A fraudulent transaction is classified as legitimate. Cost = $200.
        
        False Positive (FP): A legitimate transaction is classified as fraudulent. Cost = $5.

### The model should predict fraud when the expected cost of flagging the transaction is lower than the expected cost of allowing it.

### Let 
p be the predicted probability of fraud.

Cost of predicting fraud: (1−p)×$5

Cost of predicting legitimate: p×$200

Predict fraud when: (1−p) 5 < 200p

5 < 205 p

p> 5 / 205  p ≈ 0.0244
​
 
Therefore, the initial cost-based threshold is 2.44%. Transactions with a predicted fraud probability above 2.44% should be flagged for review.

# B Assess Score Calibration
    - Reliability diagram: Compare the fraud probabilities predicted by the model with the actual fraud rates.
    
    - Brier score: Measures how close the predicted probabilities are to the actual results. A lower score is better.
    
    - Delayed labels: Since fraud labels take 30–60 days to arrive, evaluate the model using confirmed labels once they are available.


# 5. Detect Production Drift and Decide When to Retrain
# A. Detect production drift
    1. Data drift: Check whether transaction data changes, such as changes in transaction amounts, locations, or spending habits.
    
    2. Prediction drift: Check whether the model starts flagging more or fewer transactions as fraudulent.
    
    3. Concept drift: Check whether the model's performance gets worse when actual fraud labels become available. Monitor precision, recall, and financial losses.
# B. Decide when to retrain
    - Monitor the model daily or weekly.
    
    - Investigate if transaction patterns change significantly or model performance decreases.
    
    - Retrain the model if these changes continue and affect its performance.
    
    - Test the new model on recent data before replacing the old model.
    
    - Monitor the new model after deployment and roll back if it performs poorly.